## Simple RAG with Test-Time Compute
Here we use Test-Time Compute (TTC) to execute the RAG agent multiple times and select the best output.

The workflow:
1. Executes the ReAct agent multiple times
2. Uses an LLM-based selection strategy to merge/select the best output

**Prerequisites:**
- Milvus server running at `localhost:19530`
- Collections named `cuda_docs` and `mcp_docs` with embedded documents


In [ ]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath("../../../src/")
if module_path not in sys.path:
    sys.path.insert(0, module_path)


In [ ]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)


In [ ]:
from pydantic import HttpUrl

from nat.agent.react_agent.register import NatReActAgent
from nat.embedder.nim_embedder import NIMEmbedder
from nat.experimental.test_time_compute.functions.execute_score_select_function import ExecuteScoreSelectFunction
from nat.experimental.test_time_compute.models.selection_config import LLMBasedOutputMerging
from nat.llm.nim_llm import NimLLM
from nat.retriever.milvus.register import MilvusRetriever
from nat.tool.retriever import NatRetrieverTool
from nat.utils.sdk.nat_workflow import NatWorkflow

llm = NimLLM(
    model_name="nvdev/meta/llama-3.3-70b-instruct",
    temperature=0.6,  # Higher temperature for diverse outputs
    max_tokens=4096,
    top_p=1.0,
    name="nim_llm",
)

milvus_embedder = NIMEmbedder(
    model_name="nvidia/nv-embedqa-e5-v5",
    truncate="END",
    name="milvus_embedder",
)

cuda_retriever = MilvusRetriever(
    uri=HttpUrl("http://localhost:19530"),
    collection_name="cuda_docs",
    embedder=milvus_embedder,
    top_k=10,
    name="cuda_retriever",
)

mcp_retriever = MilvusRetriever(
    uri=HttpUrl("http://localhost:19530"),
    collection_name="mcp_docs",
    embedder=milvus_embedder,
    top_k=10,
    name="mcp_retriever",
)

cuda_retriever_tool = NatRetrieverTool(
    nat_retriever=cuda_retriever,
    topic="Retrieve documentation for NVIDIA's CUDA library",
    name="cuda_retriever_tool",
)

mcp_retriever_tool = NatRetrieverTool(
    nat_retriever=mcp_retriever,
    topic="Retrieve information about Model Context Protocol (MCP)",
    name="mcp_retriever_tool",
)

# Create the ReAct agent that will be executed multiple times
react_agent = NatReActAgent(
    tools=[cuda_retriever_tool, mcp_retriever_tool],
    llm=llm,
    verbose=True,
    name="react_agent_executor",
)

# Create the LLM-based output merging strategy
selection_strategy = LLMBasedOutputMerging(
    llm=llm,
    name="selection_strategy",
)

# Create the Execute-Score-Select workflow
ttc_workflow = ExecuteScoreSelectFunction(
    augmented_function=react_agent,
    nat_selector=selection_strategy,
    num_executions=3,
    name="ttc_workflow",
)

nat_workflow = NatWorkflow(
    entrypoint=ttc_workflow,
)


In [ ]:
await nat_workflow.prompt('How do I install CUDA?')


In [ ]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config_ttc.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())
